# Heart Disease Prediction
## ML Deployment Project

---

## 1. Problem Definition

### ปัญหาคืออะไร?
โรคหัวใจ (Cardiovascular Disease) เป็นสาเหตุการเสียชีวิตอันดับ 1 ของโลก คร่าชีวิตประมาณ 17.9 ล้านคนต่อปี (WHO) การตรวจพบความเสี่ยงตั้งแต่เนิ่นๆ สามารถช่วยให้ผู้ป่วยได้รับการรักษาทันเวลาและลดอัตราการเสียชีวิตได้

### ทำไมถึงน่าแก้ไข?
- การวินิจฉัยโรคหัวใจในปัจจุบันต้องอาศัยการตรวจหลายขั้นตอนที่มีค่าใช้จ่ายสูง
- ML model สามารถช่วย **คัดกรองเบื้องต้น** จากข้อมูลพื้นฐาน เช่น อายุ ความดันโลหิต คอเลสเตอรอล ผล ECG ได้
- ช่วยให้แพทย์มีเครื่องมือสนับสนุนการตัดสินใจ (Decision Support Tool) ไม่ใช่ทดแทนแพทย์

### ทำไม ML ถึงเหมาะกับปัญหานี้?
- เป็น **Binary Classification** ที่ชัดเจน (เป็นโรคหัวใจ / ไม่เป็น)
- มี features ที่เป็นตัวแปรทางการแพทย์ที่มีความสัมพันธ์กับโรคหัวใจตามหลักวิชาการ
- Dataset มีขนาดเพียงพอ (918 ตัวอย่าง) สำหรับ traditional ML
- ความสัมพันธ์ระหว่าง features กับ target อาจเป็น non-linear ซึ่ง ML จัดการได้ดีกว่า rule-based

## 2. Dataset Description

### ที่มาของข้อมูล
Dataset นี้รวบรวมจาก 5 แหล่งข้อมูลโรคหัวใจที่มีชื่อเสียง:
1. Cleveland Clinic Foundation
2. Hungarian Institute of Cardiology
3. V.A. Medical Center, Long Beach
4. University Hospital, Zurich
5. Statlog (Heart) Dataset

เผยแพร่บน [Kaggle](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction) โดยรวม 918 ตัวอย่างหลังลบข้อมูลซ้ำ

### Features ทั้งหมด 11 ตัว + 1 Target

| Feature | Type | Description | ความหมายทางการแพทย์ |
|---------|------|-------------|---------------------|
| **Age** | Numerical | อายุผู้ป่วย (ปี) | ความเสี่ยงโรคหัวใจเพิ่มขึ้นตามอายุ |
| **Sex** | Categorical (M/F) | เพศ | ผู้ชายมีความเสี่ยงสูงกว่าในช่วงอายุเท่ากัน |
| **ChestPainType** | Categorical | ประเภทอาการเจ็บหน้าอก | ASY (ไม่มีอาการ) พบในผู้ป่วยโรคหัวใจบ่อย |
| **RestingBP** | Numerical | ความดันโลหิตขณะพัก (mm Hg) | ความดันสูง = ปัจจัยเสี่ยง |
| **Cholesterol** | Numerical | คอเลสเตอรอลในเลือด (mg/dl) | ค่าสูงเกินไป = ปัจจัยเสี่ยง |
| **FastingBS** | Binary (0/1) | น้ำตาลในเลือดขณะอดอาหาร > 120 mg/dl | เบาหวานเป็นปัจจัยเสี่ยงโรคหัวใจ |
| **RestingECG** | Categorical | ผลตรวจคลื่นไฟฟ้าหัวใจขณะพัก | Normal / ST-T wave abnormality / LVH |
| **MaxHR** | Numerical | อัตราการเต้นหัวใจสูงสุดที่ทำได้ | ค่าต่ำ = อาจบ่งบอกปัญหาหัวใจ |
| **ExerciseAngina** | Binary (Y/N) | เจ็บหน้าอกขณะออกกำลังกาย | ถ้ามี = สัญญาณเตือนสำคัญ |
| **Oldpeak** | Numerical | ST depression จากการออกกำลังกาย | ค่าสูง = หัวใจขาดเลือด |
| **ST_Slope** | Categorical | ความชันของ ST segment | Flat/Down = ผิดปกติ |
| **HeartDisease** | **Target** (0/1) | 0 = ปกติ, 1 = เป็นโรคหัวใจ | - |

### ประเภท ChestPainType
- **TA** (Typical Angina): เจ็บหน้าอกแบบ classic — เจ็บเมื่อออกแรง หายเมื่อพัก
- **ATA** (Atypical Angina): เจ็บหน้าอกแบบไม่ตรงตำรา
- **NAP** (Non-Anginal Pain): เจ็บหน้าอกที่ไม่เกี่ยวกับหัวใจ
- **ASY** (Asymptomatic): ไม่มีอาการเจ็บหน้าอก — พบบ่อยในผู้ป่วยโรคหัวใจจริงๆ

## 3. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

In [ ]:
df = pd.read_csv('heart.csv')
print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# ดู unique values ของ categorical features
cat_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
for col in cat_cols:
    print(f'{col}: {df[col].unique()} (n={df[col].nunique()})')

In [ ]:
# ดู class balance ของ target
print('Target Distribution:')
print(df['HeartDisease'].value_counts())
print(f'\nRatio: {df["HeartDisease"].value_counts(normalize=True).round(3).to_dict()}')

## 4. Exploratory Data Analysis (EDA)

### 4.1 Missing Values & ค่าผิดปกติ
จาก dataset นี้ไม่มี null values แต่มี **ค่า 0 ที่น่าสงสัย** — Cholesterol = 0 และ RestingBP = 0 ซึ่งเป็นไปไม่ได้ทางการแพทย์ จึงควรถือว่าเป็น missing values

In [ ]:
# ตรวจสอบ null values
print("Null values per column:")
print(df.isnull().sum())
print(f"\nTotal null values: {df.isnull().sum().sum()}")

# ตรวจสอบค่า 0 ที่น่าสงสัย
print(f"\n--- ค่า 0 ที่ไม่สมเหตุสมผลทางการแพทย์ ---")
print(f"Cholesterol = 0: {(df['Cholesterol'] == 0).sum()} rows ({(df['Cholesterol'] == 0).mean():.1%})")
print(f"RestingBP = 0:   {(df['RestingBP'] == 0).sum()} rows ({(df['RestingBP'] == 0).mean():.1%})")

# Cholesterol = 0 เป็นไปไม่ได้ทางการแพทย์ (คนปกติอยู่ที่ ~125-200 mg/dl)
# RestingBP = 0 ก็เช่นกัน → ถือว่าเป็น missing values

### 4.2 Target Distribution (Class Balance)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
counts = df['HeartDisease'].value_counts()
colors = ['#2ecc71', '#e74c3c']
ax[0].bar(['Normal (0)', 'Heart Disease (1)'], counts.values, color=colors)
ax[0].set_title('Heart Disease Distribution')
ax[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Pie chart
ax[1].pie(counts.values, labels=['Normal', 'Heart Disease'], autopct='%1.1f%%',
          colors=colors, startangle=90, explode=(0.05, 0.05))
ax[1].set_title('Heart Disease Proportion')

plt.tight_layout()
plt.show()

print(f"Class ratio (Heart Disease / Normal): {counts[1]/counts[0]:.2f}")
print("→ Dataset ค่อนข้าง balanced ไม่จำเป็นต้องทำ oversampling/undersampling")

### 4.3 Distribution ของ Numerical Features

In [ ]:
num_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    # แยก histogram ตาม HeartDisease
    df[df['HeartDisease'] == 0][col].hist(ax=ax, bins=30, alpha=0.6, color='#2ecc71', label='Normal')
    df[df['HeartDisease'] == 1][col].hist(ax=ax, bins=30, alpha=0.6, color='#e74c3c', label='Heart Disease')
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)
    ax.legend()

# ลบ subplot ที่เหลือ
axes[5].set_visible(False)

plt.suptitle('Numerical Features Distribution by Heart Disease', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots เพื่อดู outliers และเปรียบเทียบระหว่างกลุ่ม
fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x='HeartDisease', y=col, ax=axes[i], palette=['#2ecc71', '#e74c3c'])
    axes[i].set_xticklabels(['Normal', 'Heart Disease'])
    axes[i].set_title(col)

plt.suptitle('Box Plots: Numerical Features by Heart Disease', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("สังเกต:")
print("• Age: ผู้ป่วยโรคหัวใจมีอายุเฉลี่ยสูงกว่า")
print("• MaxHR: ผู้ป่วยโรคหัวใจมี max heart rate ต่ำกว่า")
print("• Oldpeak: ผู้ป่วยโรคหัวใจมี ST depression สูงกว่า")
print("• Cholesterol: มี spike ที่ 0 จำนวนมาก (missing values)")

### 4.4 Distribution ของ Categorical Features

In [ ]:
cat_cols = ['Sex', 'ChestPainType', 'FastingBS', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    # นับจำนวนแยกตาม HeartDisease
    ct = pd.crosstab(df[col], df['HeartDisease'])
    ct.plot(kind='bar', ax=axes[i], color=['#2ecc71', '#e74c3c'], rot=0)
    axes[i].set_title(f'{col} vs Heart Disease')
    axes[i].legend(['Normal', 'Heart Disease'])
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

plt.suptitle('Categorical Features vs Heart Disease', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("สังเกต:")
print("• Sex: ผู้ชาย (M) มีสัดส่วนเป็นโรคหัวใจสูงกว่าผู้หญิง")
print("• ChestPainType: ASY (ไม่มีอาการ) มีสัดส่วนเป็นโรคหัวใจสูงมาก")
print("• ExerciseAngina: คนที่เจ็บหน้าอกตอนออกกำลังกาย (Y) มีโอกาสเป็นโรคหัวใจสูง")
print("• ST_Slope: Flat มีสัดส่วนเป็นโรคหัวใจสูงมาก, Up มักปกติ")

### 4.5 Correlation Matrix

In [ ]:
# Correlation matrix สำหรับ numerical features
num_df = df[['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak', 'HeartDisease']]

plt.figure(figsize=(10, 8))
corr = num_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=1)
plt.title('Correlation Matrix (Numerical Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Features ที่มี correlation สูงกับ HeartDisease:")
print(corr['HeartDisease'].drop('HeartDisease').sort_values(ascending=False).to_string())
print("\nสังเกต:")
print("• Oldpeak มี positive correlation สูงสุดกับ HeartDisease")
print("• MaxHR มี negative correlation สูงสุด (ยิ่ง MaxHR ต่ำ ยิ่งเสี่ยง)")
print("• Age มี positive correlation ปานกลาง")

### 4.6 สรุป EDA

**สิ่งที่ค้นพบ:**
1. **Missing values ซ่อนอยู่**: Cholesterol = 0 (~172 rows) และ RestingBP = 0 (1 row) → ต้องจัดการใน preprocessing
2. **Class balance**: ค่อนข้าง balanced (~55% Heart Disease, ~45% Normal) → ไม่จำเป็นต้อง resample
3. **Features ที่ส่งสัญญาณชัด**: ST_Slope (Flat), ExerciseAngina (Y), ChestPainType (ASY), Oldpeak สูง, MaxHR ต่ำ
4. **Cholesterol กับ RestingBP** มี correlation ต่ำกับ target — อาจเพราะ missing values (0) ทำให้ signal หาย

**แนวทาง Preprocessing:**
- แทนค่า 0 ใน Cholesterol และ RestingBP ด้วย median (แยกตาม group ถ้าเป็นไปได้)
- ใช้ Pipeline เพื่อรวม preprocessing + model เข้าด้วยกัน ป้องกัน data leakage